## Milestone 2

In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

In [2]:
# --- 1. NOVÝ DATASET ČTOUCÍ Z CSV ---
class WikiArtCSVDataset(Dataset):
    def __init__(self, csv_file, split, transform=None):
        # Načtení celého CSV
        self.df = pd.read_csv(csv_file)
        # Filtrace pouze pro požadovaný split (train, val, nebo test)
        self.df = self.df[self.df['split'] == split].reset_index(drop=True)
        self.transform = transform
        
        # Dynamické vytvoření mapování textových labelů na čísla pro zachovaných 14 tříd
        self.classes = sorted(self.df['label'].unique())
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        label_str = self.df.loc[idx, 'label']
        label = self.class_to_idx[label_str]
        
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

In [ ]:
# --- 2. DYNAMICKÁ AUGMENTACE (DATA ENHANCEMENT) ---
# Aplikuje se pouze na trénovací data pro zamezení overfittingu
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # ImageNet standard
    transforms.RandomErasing(p=0.2) # Strukturální regularizace
])

# Pro validaci a testování nesmí probíhat náhodné deformace
val_test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
# --- 3. INICIALIZACE DATALOADERŮ ---
csv_path = "dataset_split.csv"

train_dataset = WikiArtCSVDataset(csv_file=csv_path, split='train', transform=train_transform)
val_dataset = WikiArtCSVDataset(csv_file=csv_path, split='val', transform=val_test_transform)
test_dataset = WikiArtCSVDataset(csv_file=csv_path, split='test', transform=val_test_transform)

print(f"Počet trénovacích vzorků: {len(train_dataset)}")
print(f"Počet validačních vzorků: {len(val_dataset)}")
print(f"Počet testovacích vzorků: {len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

Počet trénovacích vzorků: 60729
Počet validačních vzorků: 7591
Počet testovacích vzorků: 7592


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Používám zařízení: {device}")

# 1. TRANSFER LEARNING: Načtení předtrénovaného modelu
weights = ResNet18_Weights.IMAGENET1K_V1
model = resnet18(weights=weights)

# 2. Zmrazení základních vrstev (frozen base layers)[cite: 2]
# Tímto zakážeme výpočet gradientů pro staré vrstvy, aby si síť nezničila užitečné extraktory hran a textur.
for param in model.parameters():
    param.requires_grad = False

# 3. Výměna finálního klasifikátoru
# Nová vrstva má automaticky requires_grad=True, takže se bude trénovat pouze ona.
num_ftrs = model.fc.in_features
num_classes = len(train_dataset.classes) # 14 tříd
model.fc = nn.Linear(num_ftrs, num_classes)

model = model.to(device)

# 4. POKROČILÁ OPTIMALIZACE A REGULARIZACE[cite: 2]
# Výběr pouze těch parametrů, které vyžadují gradient (naše nová fc vrstva)
trainable_params = [p for p in model.parameters() if p.requires_grad]

# AdamW obsahuje vylepšenou implementaci L2 regularizace (Weight Decay)[cite: 2]
optimizer = optim.AdamW(trainable_params, lr=1e-3, weight_decay=1e-4)

# Dynamické plánování learning ratu pomocí OneCycleLR[cite: 2]
# Max_lr je maximální hodnota, na kterou scheduler vystoupá, než začne klesat k nule.
epochs = 10
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=3e-3,
    steps_per_epoch=len(train_loader),
    epochs=epochs
)

print(f"Model připraven. Počet trénovatelných vrstev: 1 (přepsaná model.fc)")

Používám zařízení: cuda
Model připraven. Počet trénovatelných vrstev: 1 (přepsaná model.fc)
